# سبق 10 - AI ایجنٹس پروڈکشن میں

اس سبق میں آپ سیکھیں گے **پروڈکشن کے نمونے** AI ایجنٹس کے لیے Microsoft Agent Framework (Python) استعمال کرتے ہوئے۔ ہم شامل کرتے ہیں:

- **مشاہداتی قابلیت** — ایجنٹ بات چیت میں ٹائمنگ اور لاگنگ شامل کرنا  
- **تشخیص** — جواب کے معیار کو ماپنے کے لیے ایک ایوا لیو ایٹر ایجنٹ کا استعمال  
- **لاگت کا انتظام** — ٹوکن کی بہتر استعمال اور ماڈل کے انتخاب کی حکمت عملی  

منظر نامہ ایک **ٹریول ایجنٹ** کا ہے جو صارفین کو سفر کی منصوبہ بندی میں مدد دیتا ہے، جس پر نگرانی اور تشخیص کی تہہ چڑھی ہوئی ہے۔


## ترتیب


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## پیداوار کے عوامل

AI ایجنٹس کو پروٹو ٹائپس سے پیداوار میں منتقل کرنے کے لیے تین ستونوں پر باریک بینی سے توجہ دینا ضروری ہے:

1. **مشاہداتی صلاحیت** — آپ کو یہ معلوم ہونا چاہیے کہ ایجنٹ کیا کر رہا ہے، اسے کتنا وقت لگ رہا ہے، اور وہ کون سے ٹولز استعمال کر رہا ہے۔ بغیر ٹریسنگ اور لاگنگ کے، پیداوار کے مسائل کی خرابی تلاش کرنا تقریباً ناممکن ہے۔

2. **تشخیص** — خودکار معیار کی جانچ ایجنٹ کے جوابات کو وقت کے ساتھ درست، مکمل اور مددگار رکھنے کو یقینی بناتی ہے۔ ایک تشخیصی ایجنٹ متعین معیار کے خلاف جوابات کو اسکور کر سکتا ہے۔

3. **لاگت کا انتظام** — ٹوکن کے استعمال کا براہِ راست اثر لاگت پر پڑتا ہے۔ حکمت عملیوں جیسے پرامپٹ کی اصلاح، ماڈل کا انتخاب، اور کیشنگ اخراجات کو قابو میں رکھنے میں مدد دیتی ہیں بغیر معیار کے نقصان کے۔


## ایک قابل مشاہدہ ایجنٹ بنانا

ہم سفر کے آلات کی تعریف کرتے ہیں اور ایجنٹ کال کو ٹائمنگ کے ساتھ لپیٹتے ہیں تاکہ ہم تاخیر کی نگرانی کر سکیں۔ پروڈکشن میں آپ OpenTelemetry یا کسی مشابہ ٹریسنگ بیک اینڈ کے ساتھ انٹیگریٹ کریں گے۔


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## تخمینہ کاری کے نمونے

ایک عام پروڈکشن نمونہ یہ ہے کہ دوسرے ایجنٹ کو **تخمینہ کار** کے طور پر استعمال کیا جائے۔ تخمینہ کار بنیادی ایجنٹ کے جواب کو پیشگی طے شدہ معیار کے خلاف درجہ بندی کرتا ہے جیسے مکملت، درستگی، اور مددگاری۔

اس سے ممکن ہوتا ہے:
- صارفین تک جواب پہنچنے سے پہلے خودکار معیار کے دروازے
- جب پرامپٹ یا ماڈلز تبدیل ہوتے ہیں تو ریگریشن کی نشاندہی
- وقت کے ساتھ ایجنٹ کی کارکردگی کی مسلسل نگرانی


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## لاگت کے انتظام کی حکمت عملیاں

پیداواری AI ایجنٹوں کے لیے لاگت پر قابو پانا انتہائی اہم ہے۔ یہاں کلیدی حکمت عملیاں دی گئی ہیں:

| حکمت عملی | وضاحت |
|---|---|
| **پرومپٹ کی اصلاح** | سسٹم کی ہدایات کو مختصر رکھیں۔ ان پٹ ٹوکنز کو کم کرنے کے لیے غیر ضروری سیاق و سباق کو ہٹا دیں۔ |
| **ماڈل کا انتخاب** | آسان کاموں جیسے درجہ بندی یا استخراج کے لیے چھوٹے، سستے ماڈلز (مثلاً GPT-4o-mini) استعمال کریں، اور پیچیدہ استدلال کے لیے بڑے ماڈلز کو مخصوص رکھیں۔ |
| **کیچنگ** | آلے کے نتائج اور بار بار پوچھے جانے والے سوالات کو کیش کرکے غیر ضروری API کالز سے بچیں۔ |
| **ٹوکن بجٹ** | غیر متوقع طور پر طویل جوابات کو روکنے کے لیے `max_tokens` کی حدود مقرر کریں۔ |
| **بیچنگ** | جہاں ممکن ہو، متعدد صارف کے سوالات کو ایک ہی API کال میں گروپ کریں۔ |

عملی طور پر، ایک طبقہ وار طریقہ اچھی طرح کام کرتا ہے: سیدھے سادے درخواستوں کو تیز، سستے ماڈل کی طرف بھیجیں اور صرف پیچیدہ سوالات کو زیادہ قابل ماڈل تک بڑھائیں۔


## خلاصہ

اس سبق میں آپ نے یہ سیکھا کہ کیسے:

1. **ایجنٹ تعاملات میں مشاہداتی صلاحیت** شامل کی جائے وقت بندی اور لاگنگ کے ساتھ، جو ٹریسنگ اور مانیٹرنگ کی بنیاد فراہم کرتی ہے۔
2. **ایجنٹ کے ردعمل کا خودکار انداز میں جائزہ لیں** ایک جانچ کرنے والے ایجنٹ کے ذریعے جو مکمل ہونے، درستگی، اور مددگار ہونے کی درجہ بندی کرتا ہے۔
3. **لاگت کا انتظام کریں** پرامپٹ کی اصلاح، ماڈل کے انتخاب، کیشنگ، اور ٹوکن بجٹ کے ذریعے۔

یہ پیداواری طریقے اس بات کو یقینی بنانے میں مدد دیتے ہیں کہ آپ کے AI ایجنٹس قابل اعتماد، قابل پیمائش، اور بڑے پیمانے پر لاگت کے لحاظ سے موثر ہوں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
